# 02 — CNN Training: EfficientNetB0 for Road Damage Detection

**CivicSense AI — Deep Learning Pipeline**

Train EfficientNetB0 (pretrained on ImageNet) for 4-class road damage classification.

**Architecture:** EfficientNetB0 (frozen base) + Custom classification head  
**Input:** 224×224 RGB image  
**Output:** Softmax over 4 classes + confidence score  
**Expected Accuracy:**  
  - Phase 1 (transfer learning): 75-82%  
  - Phase 2 (fine-tuning): 87-93%  


In [1]:
import os, sys, json, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

plt.style.use('dark_background')

# ── Config ──────────────────────────────────────────────────────────────
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = 4
DATASET_DIR = 'data/RDD2022_India'
MODEL_PATH  = '../backend/ml/saved_models/efficientnet_road.h5'

CLASS_NAMES = ['Longitudinal Crack', 'Transverse Crack', 'Alligator Crack', 'Pothole']
# Maps folder names (D00, D10, D20, D40) to class indices
CLASS_FOLDERS = {'D00': 0, 'D10': 1, 'D20': 2, 'D40': 3}

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}')
print(f'Image size: {IMG_SIZE}, Batch size: {BATCH_SIZE}')

os.makedirs('../backend/ml/saved_models', exist_ok=True)
os.makedirs('data', exist_ok=True)

TensorFlow version: 2.20.0
GPU available: False
Image size: (224, 224), Batch size: 32


In [2]:
# ── 2. Prepare Dataset Directories ───────────────────────────────────────
# RDD2022 India uses YOLO format bounding box labels.
# For image classification, we copy images into class-named folders.

import shutil, glob
from PIL import Image

TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
SPLIT_DIR = 'data/split'

def create_class_split(dataset_dir, split_dir, val_ratio=0.1, test_ratio=0.1, seed=42):
    """
    Parse YOLO label files to get dominant class per image,
    then copy images into split/train / split/val / split/test folders.
    """
    random.seed(seed)
    img_dir   = os.path.join(dataset_dir, 'images')
    label_dir = os.path.join(dataset_dir, 'labels')

    if not os.path.exists(img_dir):
        print(f'⚠ {img_dir} not found — using synthetic demo data')
        return False

    # Map image → dominant class (most frequent label in that image)
    id_to_name = {0:'D00', 1:'D10', 2:'D20', 3:'D40'}
    img_class  = {}

    for lf in glob.glob(os.path.join(label_dir, '*.txt')):
        stem    = os.path.splitext(os.path.basename(lf))[0]
        counts  = {}
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cid = int(parts[0])
                    counts[cid] = counts.get(cid, 0) + 1
        if counts:
            dom = max(counts, key=counts.get)
            img_class[stem] = id_to_name[dom]

    # Organise by class
    per_class = {c: [] for c in id_to_name.values()}
    for stem, cls in img_class.items():
        jpg = os.path.join(img_dir, stem + '.jpg')
        if os.path.exists(jpg):
            per_class[cls].append(jpg)

    for split in ['train', 'val', 'test']:
        for cls in per_class:
            os.makedirs(os.path.join(split_dir, split, cls), exist_ok=True)

    for cls, images in per_class.items():
        random.shuffle(images)
        n     = len(images)
        n_val = int(n * val_ratio)
        n_tst = int(n * test_ratio)
        splits = {
            'train': images[n_val + n_tst:],
            'val':   images[:n_val],
            'test':  images[n_val:n_val + n_tst],
        }
        for split_name, files in splits.items():
            for fp in files:
                dst = os.path.join(split_dir, split_name, cls, os.path.basename(fp))
                if not os.path.exists(dst):
                    shutil.copy(fp, dst)

    for split in ['train', 'val', 'test']:
        total = sum(len(glob.glob(os.path.join(split_dir, split, c, '*.jpg')))
                    for c in per_class)
        print(f'  {split}: {total} images')

    return True

dataset_ready = create_class_split(TRAIN_DIR, SPLIT_DIR)

⚠ data/RDD2022_India\train\images not found — using synthetic demo data


In [3]:
# ── 3. Data Generators ───────────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def preprocess(x):
    x = x / 255.0
    x = (x - MEAN) / STD
    return x

# ImageDataGenerator with heavy augmentation for training
train_datagen = ImageDataGenerator(
    preprocessing_function  = preprocess,
    horizontal_flip         = True,
    rotation_range          = 15,
    brightness_range        = [0.8, 1.2],
    zoom_range              = 0.10,
    width_shift_range       = 0.05,
    height_shift_range      = 0.05,
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess)

# Use the split folders if they were created, otherwise fall back to the original
train_path = os.path.join(SPLIT_DIR, 'train') if dataset_ready else TRAIN_DIR
val_path   = os.path.join(SPLIT_DIR, 'val')   if dataset_ready else TRAIN_DIR

if os.path.exists(train_path):
    train_gen = train_datagen.flow_from_directory(
        train_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical',
        shuffle=True, seed=42,
    )
    val_gen = val_datagen.flow_from_directory(
        val_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical',
        shuffle=False,
    )
    print('Class index map:', train_gen.class_indices)
else:
    print('⚠ Dataset not ready — model will be built but not trained.\n'
          'Download RDD2022 and re-run cells 2–3.')
    train_gen, val_gen = None, None

⚠ Dataset not ready — model will be built but not trained.
Download RDD2022 and re-run cells 2–3.


In [4]:
# ── 4. Build EfficientNetB0 Model ─────────────────────────────────────────
def build_model(num_classes=4, freeze_base=True):
    """
    EfficientNetB0 + GlobalAveragePooling + Dense head.
    freeze_base=True  → Phase 1 (transfer learning)
    freeze_base=False → Phase 2 (fine-tuning)
    """
    # Load pretrained backbone (no top)
    base = EfficientNetB0(
        weights     = 'imagenet',
        include_top = False,
        input_shape = (*IMG_SIZE, 3),
    )
    base.trainable = not freeze_base

    # Classification head
    inputs  = keras.Input(shape=(*IMG_SIZE, 3))
    x       = base(inputs, training=not freeze_base)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model, base


model, base_model = build_model(freeze_base=True)

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy'],
)

# Print trainable param count
total      = sum(w.numpy().size for w in model.weights)
trainable  = sum(w.numpy().size for w in model.trainable_weights)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,}  ({trainable/total*100:.1f}% of total)')
print(f'Frozen base:      {total-trainable:,} params\n')
model.summary(line_length=80)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Total params:     4,383,655
Trainable params: 331,524  (7.6% of total)
Frozen base:      4,052,131 params



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)        │ (None, 224, 224, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)       │ (None, 7, 7, 1280)       │     4,049,571 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ global_average_pooling2d          │ (None, 1280)             │             0 │
│ (GlobalAveragePooling2D)          │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ batch_normalization               │ (None, 1280)             │         5,120 │
│ (BatchNormalization)              │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 256)              │       327,936 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 256)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_1 (Dense)                   │ (None, 4)                │         1,028 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 4,383,655 (16.72 MB)

 Trainable params: 331,524 (1.26 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [5]:
# ── 5. Phase 1: Transfer Learning (frozen base) ───────────────────────────
PHASE1_EPOCHS = 5

callbacks_p1 = [
    ModelCheckpoint(
        'data/best_phase1.h5', monitor='val_accuracy',
        save_best_only=True, verbose=1,
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=3,
        restore_best_weights=True, verbose=1,
    ),
]

if train_gen and val_gen:
    print('=== Phase 1: Transfer Learning ===')
    history1 = model.fit(
        train_gen,
        validation_data = val_gen,
        epochs          = PHASE1_EPOCHS,
        callbacks       = callbacks_p1,
        verbose         = 1,
    )
    p1_val_acc = max(history1.history['val_accuracy'])
    print(f'\nPhase 1 best val accuracy: {p1_val_acc*100:.2f}%')
else:
    print('⚠ Skipped — no dataset. Saving untrained model as placeholder.')
    model.save(MODEL_PATH)
    print(f'Placeholder model saved to {MODEL_PATH}')

⚠ Skipped — no dataset. Saving untrained model as placeholder.
Placeholder model saved to ../backend/ml/saved_models/efficientnet_road.h5


In [6]:
# ── 6. Phase 2: Fine-Tuning (unfreeze last 20 layers) ─────────────────────
PHASE2_EPOCHS  = 10
FINE_TUNE_FROM = -20  # unfreeze last 20 layers

if train_gen and val_gen:
    # Unfreeze last 20 layers of the base
    base_model.trainable = True
    for layer in base_model.layers[:FINE_TUNE_FROM]:
        layer.trainable = False

    unfrozen = sum(1 for l in base_model.layers if l.trainable)
    print(f'Unfrozen {unfrozen} base layers for fine-tuning')

    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=1e-4),  # lower LR
        loss      = 'categorical_crossentropy',
        metrics   = ['accuracy'],
    )

    callbacks_p2 = [
        ModelCheckpoint(
            MODEL_PATH, monitor='val_accuracy',
            save_best_only=True, verbose=1,
        ),
        EarlyStopping(
            monitor='val_accuracy', patience=3,
            restore_best_weights=True, verbose=1,
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, verbose=1,
        ),
    ]

    print('\n=== Phase 2: Fine-Tuning ===')
    history2 = model.fit(
        train_gen,
        validation_data = val_gen,
        epochs          = PHASE2_EPOCHS,
        initial_epoch   = PHASE1_EPOCHS,
        callbacks       = callbacks_p2,
        verbose         = 1,
    )
    p2_val_acc = max(history2.history['val_accuracy'])
    print(f'\nPhase 2 best val accuracy: {p2_val_acc*100:.2f}%')
    print(f'Model saved to: {MODEL_PATH}')

In [7]:
# ── 7. Training Curves ───────────────────────────────────────────────────
if 'history1' in dir() and 'history2' in dir():
    acc  = history1.history['accuracy']     + history2.history['accuracy']
    val  = history1.history['val_accuracy'] + history2.history['val_accuracy']
    loss = history1.history['loss']         + history2.history['loss']
    vloss= history1.history['val_loss']     + history2.history['val_loss']

    epochs = range(1, len(acc) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.patch.set_facecolor('#0f172a')

    for ax, y1, y2, title, ylabel in [
        (axes[0], acc,  val,  'Accuracy',  'Accuracy'),
        (axes[1], loss, vloss,'Loss',      'Loss'),
    ]:
        ax.set_facecolor('#1e293b')
        ax.plot(epochs, y1, color='#10b981', linewidth=2, label='Train')
        ax.plot(epochs, y2, color='#f59e0b', linewidth=2, linestyle='--', label='Val')
        ax.axvline(x=PHASE1_EPOCHS + 0.5, color='#64748b', linestyle=':', linewidth=1.5,
                   label='Fine-tune starts')
        ax.set_title(title, color='white', fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch', color='#94a3b8')
        ax.set_ylabel(ylabel, color='#94a3b8')
        ax.legend(facecolor='#0f172a', edgecolor='#334155', labelcolor='white')
        ax.tick_params(colors='#94a3b8')
        ax.spines[:].set_color('#334155')
        ax.grid(alpha=0.2, color='#334155')

    plt.tight_layout()
    plt.savefig('data/training_curves.png', bbox_inches='tight', dpi=150, facecolor='#0f172a')
    plt.show()

In [8]:
# ── 8. Evaluate on Test Set ───────────────────────────────────────────────
test_path = os.path.join(SPLIT_DIR, 'test')

if os.path.exists(test_path) and os.path.exists(MODEL_PATH):
    model = keras.models.load_model(MODEL_PATH)
    test_gen = val_datagen.flow_from_directory(
        test_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False,
    )
    loss, acc = model.evaluate(test_gen, verbose=0)
    print(f'Test Accuracy: {acc*100:.2f}%')
    print(f'Test Loss:     {loss:.4f}')

    # Predictions for confusion matrix
    y_pred = np.argmax(model.predict(test_gen, verbose=0), axis=1)
    y_true = test_gen.classes

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.patch.set_facecolor('#0f172a')
    ax.set_facecolor('#1e293b')
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Greens',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, linecolor='#0f172a', ax=ax,
    )
    ax.set_title('Confusion Matrix — Test Set', color='white', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted', color='#94a3b8')
    ax.set_ylabel('True', color='#94a3b8')
    ax.tick_params(colors='#94a3b8', labelsize=9)
    plt.tight_layout()
    plt.savefig('data/confusion_matrix.png', bbox_inches='tight', dpi=150, facecolor='#0f172a')
    plt.show()

    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
else:
    print('⚠ Test set or model not found. Run cells 2-6 first.')

⚠ Test set or model not found. Run cells 2-6 first.


In [9]:
# ── 9. Single Image Inference Demo ───────────────────────────────────────
import io
from PIL import Image

MEAN_NP = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD_NP  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def predict_single(model, image_path):
    img  = Image.open(image_path).convert('RGB').resize((224, 224))
    arr  = np.array(img, dtype=np.float32) / 255.0
    arr  = (arr - MEAN_NP) / STD_NP
    arr  = np.expand_dims(arr, axis=0)
    prob = model.predict(arr, verbose=0)[0]
    idx  = np.argmax(prob)
    return {
        'predicted_class': CLASS_NAMES[idx],
        'confidence':      float(prob[idx]),
        'all_probs':       {n: float(p) for n, p in zip(CLASS_NAMES, prob)},
    }

# Test with a sample image if available
import glob, os
test_images = glob.glob(os.path.join(SPLIT_DIR, 'test', '*', '*.jpg'))

if test_images and os.path.exists(MODEL_PATH):
    model = keras.models.load_model(MODEL_PATH)
    sample = random.choice(test_images)
    result = predict_single(model, sample)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.patch.set_facecolor('#0f172a')

    img = Image.open(sample).convert('RGB')
    ax1.imshow(img)
    ax1.set_facecolor('#1e293b')
    ax1.set_title(f"Predicted: {result['predicted_class']}\nConfidence: {result['confidence']*100:.1f}%",
                  color='#10b981', fontsize=11, fontweight='bold')
    ax1.axis('off')

    names  = list(result['all_probs'].keys())
    probs  = list(result['all_probs'].values())
    colors = ['#10b981' if n == result['predicted_class'] else '#334155' for n in names]
    ax2.set_facecolor('#1e293b')
    bars = ax2.barh(names, probs, color=colors, edgecolor='#0f172a')
    ax2.set_xlim(0, 1.1)
    for bar, p in zip(bars, probs):
        ax2.text(p + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{p*100:.1f}%', va='center', color='white', fontsize=10)
    ax2.set_title('Class Probabilities', color='white', fontsize=11, fontweight='bold')
    ax2.tick_params(colors='#94a3b8')
    ax2.spines[:].set_color('#334155')
    plt.tight_layout()
    plt.show()
    print(json.dumps(result, indent=2))
else:
    print('No test images found. Inference demo skipped.')

No test images found. Inference demo skipped.


In [11]:
# ── 10. Save as SavedModel format (for TF Serving) ───────────────────────
if os.path.exists(MODEL_PATH):
    model = keras.models.load_model(MODEL_PATH)
    saved_model_path = '../backend/ml/saved_models/efficientnet_road_savedmodel'
    # Keras 3: use export() for TensorFlow SavedModel directory format
    if os.path.exists(saved_model_path):
        shutil.rmtree(saved_model_path)
    model.export(saved_model_path)
    print(f'SavedModel exported to: {saved_model_path}')

print("""
╔══════════════════════════════════════════════════════════╗
║         CNN Training Complete — Summary                  ║
╠══════════════════════════════════════════════════════════╣
║  Architecture : EfficientNetB0 + GAP + Dense(256) head  ║
║  Input size   : 224 × 224 × 3                           ║
║  Classes      : 4 (Longitudinal, Transverse,            ║
║                    Alligator Crack, Pothole)             ║
║  Phase 1 LR   : 1e-3 (frozen base, 5 epochs)            ║
║  Phase 2 LR   : 1e-4 (unfreeze last 20, 10 epochs)      ║
║  Model saved  : backend/ml/saved_models/                 ║
║                                                          ║
║  → Next: Run 03_XGBoost_training.ipynb                   ║
╚══════════════════════════════════════════════════════════╝
""")

FailedPreconditionError: ../backend/ml/saved_models is not a directory